# NASA Airfoil Self-Noise Prediction
Predict the scaled sound pressure level (SSPL, dB) of an airfoil from five physical inputs using regression models.

**Plan:** EDA -> split first -> preprocessing in a Pipeline (no data leakage) -> Linear / Ridge / Lasso / Random Forest / Gradient Boosting -> cross-validation -> tuning -> final test.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 1. Load the data

In [ ]:
df = pd.read_csv("../data/AirfoilSelfNoise.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

## 2. Exploratory data analysis

In [ ]:
print(df.describe())
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.hist(figsize=(10, 7), bins=30)
plt.tight_layout()
plt.show()

Skewness: values above about 0.5 are noticeably right-skewed.

In [ ]:
print(df.skew())

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.show()

**Observations:** `f` (2.14) and `delta` (1.70) are strongly right-skewed, `alpha` (0.69) is mildly skewed. `alpha` and `delta` are correlated (0.75). Correlations with the target are all weak, so linear models are expected to struggle.

## 3. Train/test split (done first to avoid data leakage)

In [ ]:
X = df.drop(columns="SSPL")
y = df["SSPL"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

## 4. Preprocessing pipeline
- `f`: `log1p` then scale (large values, strong skew)
- `delta`: `log` then scale (values are tiny, so `log1p` would barely change them; all values are > 0)
- `alpha`: `sqrt` then scale (contains zeros)
- `c`, `U_infinity`: scale only

Everything is fitted on the training data only.

In [ ]:
preprocess = ColumnTransformer([
    ("log1p", Pipeline([("t", FunctionTransformer(np.log1p)), ("s", StandardScaler())]), ["f"]),
    ("log",   Pipeline([("t", FunctionTransformer(np.log)),   ("s", StandardScaler())]), ["delta"]),
    ("sqrt",  Pipeline([("t", FunctionTransformer(np.sqrt)),  ("s", StandardScaler())]), ["alpha"]),
    ("scale", StandardScaler(), ["c", "U_infinity"]),
])

# Quick check that the transforms reduce skew
print("f     :", df["f"].skew().round(2), "->", np.log1p(df["f"]).skew().round(2))
print("delta :", df["delta"].skew().round(2), "->", np.log(df["delta"]).skew().round(2))
print("alpha :", df["alpha"].skew().round(2), "->", np.sqrt(df["alpha"]).skew().round(2))

## 5. Compare five models on the test set

In [ ]:
models = {
    "Linear":            Pipeline([("prep", preprocess), ("m", LinearRegression())]),
    "Ridge":             Pipeline([("prep", preprocess), ("m", Ridge(alpha=1.0))]),
    "Lasso":             Pipeline([("prep", preprocess), ("m", Lasso(alpha=0.01))]),
    "Random Forest":     RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "MAE":  mean_absolute_error(y_test, p),
        "R2":   r2_score(y_test, p),
    })

pd.DataFrame(rows).round(3)

## 6. 5-fold cross-validation (training data only)

In [ ]:
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring="r2")
    print(f"{name:18s} mean R2: {scores.mean():.3f}  (+/- {scores.std():.3f})")

## 7. Hyperparameter tuning (training data only)

In [ ]:
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"n_estimators": [200, 500],
     "learning_rate": [0.05, 0.1],
     "max_depth": [3, 5]},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gb_grid.fit(X_train, y_train)
print("Best params:", gb_grid.best_params_)
print("CV RMSE:", round(-gb_grid.best_score_, 3))

In [ ]:
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {"n_estimators": [200, 500],
     "max_depth": [None, 20],
     "min_samples_leaf": [1, 2]},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print("Best params:", rf_grid.best_params_)
print("CV RMSE:", round(-rf_grid.best_score_, 3))

## 8. Final evaluation on the test set (used once)

In [ ]:
for name, g in [("Tuned Gradient Boosting", gb_grid), ("Tuned Random Forest", rf_grid)]:
    p = g.predict(X_test)
    print(name, "RMSE:", round(np.sqrt(mean_squared_error(y_test, p)), 3),
          "R2:", round(r2_score(y_test, p), 3))

## 9. Feature importance

In [ ]:
best_rf = rf_grid.best_estimator_
imp = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(title="Random Forest - Feature Importance")
plt.xlabel("Importance")
plt.show()

In [ ]:
best_gb = gb_grid.best_estimator_
imp_gb = pd.Series(best_gb.feature_importances_, index=X.columns).sort_values()
imp_gb.plot.barh(title="Gradient Boosting - Feature Importance")
plt.xlabel("Importance")
plt.show()

## 10. Predicted vs actual (tuned Gradient Boosting)

In [ ]:
p = best_gb.predict(X_test)
plt.scatter(y_test, p, s=8, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual SSPL (dB)")
plt.ylabel("Predicted SSPL (dB)")
plt.title("Predicted vs Actual")
plt.show()

## 11. Conclusion
Linear models (R2 about 0.51) underfit because the relationship between the inputs and the noise level is nonlinear. Tree ensembles capture it: the tuned Gradient Boosting model gave the best held-out result (RMSE about 1.5 dB, R2 about 0.95), followed by Random Forest (RMSE about 1.8 dB, R2 about 0.935). Frequency and suction-side displacement thickness are the most influential features. Preprocessing was fitted inside Pipelines on training data only, and all tuning used cross-validation on the training set, so the test set was used once.